# Understanding Cointegration - Visual Guide

This notebook explains **what cointegration is** and **how we test for it** using interactive visualizations.

## Key Concepts
1. **Correlation vs Cointegration** - What's the difference?
2. **Linear Regression** - Finding the relationship (hedge ratio)
3. **The Spread** - Residuals from regression
4. **Stationarity** - Does the spread mean-revert?
5. **ADF Test** - Statistical test for stationarity

In [ ]:
import sys
from pathlib import Path

# Add src to path
project_root = Path.cwd().parent
sys.path.insert(0, str(project_root / "src"))
sys.path.insert(0, str(project_root))

import numpy as np
from statistical_arbitrage.visualization.educational_plots import (
    plot_cointegration_concept,
    plot_regression_explained,
    plot_adf_test_explained,
)

## 1. Correlation vs Cointegration

### Your Understanding (Correct!):
- **Linear Regression**: Model the relationship between two assets
- **Spread = Error**: Difference between actual and fitted values
- **If spread is stationary**: Assets are cointegrated
- **Hedge ratio = β**: The slope from regression

### The Visual Difference:

In [ ]:
# This creates two examples:
# Top: Correlated but NOT cointegrated (spread drifts away)
# Bottom: Cointegrated (spread mean-reverts)

fig = plot_cointegration_concept()
fig.show()

### Key Insight:

**❌ Correlation alone is not enough!**
- Assets can be 90% correlated but still diverge forever
- The spread can keep drifting in one direction

**✅ Cointegration means the spread is "tethered"**
- Like a dog on a leash - it can wander but must return
- This is what makes pairs trading possible!

---

## 2. The Linear Regression Step

### In Your Own Words (Spot On!):
> "We assume there is a linear relationship which we model with a linear regression.
> Then we see what the error is of the linear regression. This error is the spread."

### The Math:
```
ETH = β × ETC + α + ε
```

Where:
- **β** = Hedge ratio (slope) - "How much ETC equals 1 ETH?"
- **α** = Intercept (constant)
- **ε** = Residual (the SPREAD)

### Example with Real-ish Numbers:

In [ ]:
# Simulate example data
np.random.seed(42)
etc_prices = np.linspace(15, 20, 100) + np.random.randn(100) * 0.5
eth_prices = 170 * etc_prices + 500 + np.random.randn(100) * 100  # ETH ≈ 170 × ETC + noise

# Run regression
coeffs = np.polyfit(etc_prices, eth_prices, 1)
hedge_ratio = coeffs[0]  # β
intercept = coeffs[1]    # α

print(f"Regression Result:")
print(f"  Hedge Ratio (β): {hedge_ratio:.2f}")
print(f"  Intercept (α): {intercept:.2f}")
print(f"\nEquation: ETH = {hedge_ratio:.2f} × ETC + {intercept:.2f}")
print(f"\nMeaning: 1 ETH ≈ {hedge_ratio:.2f} units of ETC")

In [ ]:
# Visualize the regression
fig = plot_regression_explained(etc_prices, eth_prices, hedge_ratio, intercept)
fig.show()

### What You're Looking At:

**Left Plot:**
- Blue dots = Actual (ETH, ETC) price pairs
- Red line = Best fit line (the relationship)
- Orange dotted lines = Residuals (vertical distance from line)

**Right Plot:**
- The residuals plotted over time
- This is the **SPREAD**
- Question: Does it return to the mean (green line)?

---

## 3. The ADF Test - How We Check Stationarity

### Your Question:
> "How do we go from a delta spread to a p-value and test statistic?"

### The Answer:

The ADF test checks: **"Does past spread predict future changes?"**

#### If spread is STATIONARY (mean-reverting):
```
When spread is HIGH → next change is NEGATIVE (reverts down)
When spread is LOW → next change is POSITIVE (reverts up)
```

#### The regression the ADF test runs:
```
Δspread(t) = β × spread(t-1) + noise

If β < 0: Mean-reverting! ✅
If β ≈ 0: Random walk ❌
```

Let's visualize this:

In [ ]:
# Calculate the spread from our example
spread = eth_prices - hedge_ratio * etc_prices

# Visualize the ADF test
fig = plot_adf_test_explained(spread)
fig.show()

### What You're Looking At:

**Top Left:** The spread over time
- Does it oscillate around the mean (green line)?

**Top Right:** Changes in spread (Δspread)
- The differences: spread(t) - spread(t-1)

**Bottom Left:** The KEY plot!
- X-axis: Previous spread value
- Y-axis: Next change in spread
- **Negative slope** = Mean reversion (when high, goes down)
- **Flat slope** = Random walk (no pattern)

**Bottom Right:** Interpretation table
- Shows the test results

---

## 4. Understanding the Statistics

### Test Statistic

From the regression `Δspread = β × spread(t-1) + noise`:

```python
test_statistic = β / standard_error(β)
```

**Interpretation:**
- More negative = Stronger mean reversion
- Example: **-4.5** is very strong, **-2.0** is weak

### Critical Values (from Statistical Tables)

Pre-calculated thresholds from **thousands of simulations** of random walks:

```
1%:  -3.43  (very strict - 99% confidence)
5%:  -2.86  (standard - 95% confidence)
10%: -2.57  (lenient - 90% confidence)
```

**Decision Rule:**
```
If test_statistic < critical_value:
    "Reject null hypothesis" → Stationary! ✅
else:
    "Fail to reject" → Not stationary ❌
```

**Example:**
```
Test statistic: -3.85
Critical (5%):  -2.86

-3.85 < -2.86 → YES, stationary!
```

### P-Value

**Definition:** Probability of seeing this data if the spread was truly a random walk

```
p = 0.001 → "Only 0.1% chance this is random" → Very strong evidence ✅
p = 0.03  → "Only 3% chance this is random" → Good evidence ✅
p = 0.10  → "10% chance this is random" → Weak evidence ⚠️
p = 0.50  → "50% chance this is random" → No evidence ❌
```

**Standard threshold:** p < 0.05 (95% confidence)

---

## 5. Putting It All Together

### The Complete Process:

```
1. Linear Regression:
   ETH = β × ETC + α
   → Get hedge ratio (β)

2. Calculate Spread:
   spread = ETH - β × ETC
   → These are the residuals

3. ADF Test on Spread:
   Δspread(t) = λ × spread(t-1) + noise
   → Get test statistic and p-value

4. Decision:
   IF p < 0.05:
       "Spread is stationary"
       "Assets are COINTEGRATED" ✅
       "Can do pairs trading!"
   ELSE:
       "Spread is NOT stationary"
       "Assets are NOT cointegrated" ❌
       "Don't trade this pair"
```

### Your ETH/ETC Results:

From notebook 02:
```
P-value: 0.35 (> 0.05)
Test Statistic: -2.34 (> -2.86 critical value)

Conclusion: NOT cointegrated ❌
```

**What this means:**
- ETH and ETC are somewhat correlated (0.52)
- But their spread is NOT mean-reverting
- The spread can drift away indefinitely
- **Don't trade this pair!**

---

## Summary

### Visual Learner Recap:

1. **📊 Scatter Plot** (regression) → Find the relationship
2. **📈 Spread Chart** → Plot the residuals over time
3. **🔄 Δspread vs spread(t-1)** → Check for mean reversion pattern
4. **📉 Negative slope?** → Stationary! (cointegrated)
5. **📐 Flat slope?** → Not stationary (not cointegrated)

### Next Steps:

- Run notebook 02 with real ETH/ETC data
- See all these visualizations with actual prices
- Try different cryptocurrency pairs
- Find a pair that IS cointegrated!

### Suggested Pairs to Test:
- BTC/ETH (two dominant coins)
- USDT/USDC (stablecoins - likely cointegrated!)
- ETH/BNB (competing layer-1s)

---

**Questions?** The math is doing exactly what you described:
1. Regression → relationship
2. Residuals → spread
3. Test if spread is constant → cointegration

You understood it perfectly! 🎯